# Session 4, German credit: build it, then interrogate it

> **Your brief: build a classifier that decides who gets a loan, then ask whether it is fair.**

A bank has 1,000 past applicants, each labelled a **good** or **bad** credit risk. You'll build a model that predicts the label. That part is easy. The real work is the question Claude won't ask on its own: **is the model fair, and could you defend it to a compliance officer?**

**How to work**
- Solo or in pairs. Let Claude write the code; *you* interrogate what it produces.
- A model you can't break is not the same as a model that is right, or fair.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session4/data/german_credit.csv"

import pandas as pd

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload german_credit.csv ...")
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    except ImportError:
        df = pd.read_csv("data/german_credit.csv")   # local fallback (repo: data/german_credit.csv)

df.head()

## What the columns mean

1,000 applicants, 20 attributes + the outcome. Most categorical fields use the original Statlog codes (e.g. `A34`); the numeric fields are the useful ones for today.

| Column | Meaning |
|---|---|
| `age` | applicant age in years |
| `credit_amount` | size of the loan requested |
| `duration` | loan term, months |
| `credit_history`, `checking`, `savings`, `employment_since`, `housing`, `purpose`, … | coded categorical attributes |
| `target` | **1 = bad credit risk, 0 = good credit risk** |

> For a full code table, ask Claude: *"Explain the Statlog German credit attribute codes (A11, A34, A43 …) in plain English."*

## Step 1, Build a classifier (let Claude drive)

> Ask Claude: *"Build a logistic regression predicting `target` from the other columns. One-hot encode the categoricals, split 70/30 train/test, and report the accuracy."*

Run the code it gives you and **note the accuracy**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = pd.get_dummies(df.drop(columns='target'), drop_first=True)
y = df['target']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
print('Accuracy:', round(accuracy_score(yte, clf.predict(Xte)), 3))

You should get a respectable accuracy. Ask Claude if this is a good model. **It will probably say yes.** Hold that thought, we're about to ask a different question.

> ⚠️ *A note on the baseline:* 70% of applicants are 'good'. What accuracy would you get by predicting **everyone is good**? Compare, and keep it in mind for Session 5.

## Step 2, Read the model: one coefficient as an odds ratio

Before you judge the model, *read* it. A logistic model gives each feature a coefficient in **log-odds**; take **e to the power** of it and you get an **odds ratio**, a plain multiplier on the odds of the outcome. Here the outcome is `target = 1`, i.e. **flagged a bad credit risk**, so an odds ratio **> 1** *raises* the odds of being flagged bad, **< 1** *lowers* them, and **= 1** means no effect.

> Ask Claude: *"Turn my logistic model's coefficients into odds ratios and show the features that move the odds the most."*

In [ ]:
import numpy as np

odds_ratios = pd.Series(np.exp(clf.coef_[0]), index=X.columns)
print('Raise the odds of BAD the most:')
print(odds_ratios.sort_values(ascending=False).head(5).round(2))
print('\nLower the odds of BAD the most:')
print(odds_ratios.sort_values().head(5).round(2))
print(f"\nAge: {odds_ratios['age']:.3f} per year")

**Now say one out loud, in a single sentence**, the exact skill from the slide. Pick a feature with a meaningful unit (e.g. **age** per year, or loan **duration** per month; skip `credit_amount`, whose per-unit ratio sits near 1.00 because one euro is tiny).

Age is the one to watch: its odds ratio is **below 1**, so each extra year *lowers* the odds of being flagged a bad risk. Put that into words for a 25-year-old versus a 55-year-old, and notice you are already looking at the fairness question, straight out of the model's own coefficients.

In [ ]:
# Your one-sentence reading of the age odds ratio:
#

## Step 3, Predict before you probe  🎯

Two applicants, **identical** financials, same loan, same history, same everything, except one is **25** and the other is **55**.

- Do you expect the model to treat them the same?
- If not, which way do you expect it to lean, and why?

In [ ]:
# Your prediction before you look:
#
# Will the model treat the 25yo and 55yo the same?  :
# If not, who does it favour, and why?              :

## Step 4, Run the probe

Take a real applicant, copy them, and change **only the age**. Compare the predicted probability of being flagged a bad risk.

> Ask Claude: *"Take one applicant, make two copies identical except age = 25 vs age = 55, and show the model's predicted probability of 'bad credit' for each."*

In [ ]:
import numpy as np

base = X.iloc[[89]].copy()   # applicant 89: a small loan (1,108), 12 months, for repairs
young = base.copy(); old = base.copy()
young['age'] = 25; old['age'] = 55

p_young = clf.predict_proba(young)[0, 1]
p_old   = clf.predict_proba(old)[0, 1]
print(f'P(bad risk) at age 25: {p_young:.3f}  ->', 'DECLINE' if p_young >= 0.5 else 'approve')
print(f'P(bad risk) at age 55: {p_old:.3f}  ->', 'DECLINE' if p_old >= 0.5 else 'approve')

Same person, same loan, same history. Only the age changed, and the decision flipped. Is applicant 89 a one-off? Count how many of the 1,000 applicants would flip the same way:

In [ ]:
as25 = X.copy(); as25['age'] = 25
as55 = X.copy(); as55['age'] = 55
flips = (clf.predict_proba(as25)[:, 1] >= 0.5) & (clf.predict_proba(as55)[:, 1] < 0.5)
print(f'{flips.sum()} of {len(X)} applicants: declined at 25, approved at 55, nothing else changed')

Now widen the lens. Ask Claude to compare the model across **age groups**, not just one applicant:

> *"Split the test set into under-25, 25–54 and 55+. For each group, show the share the model flags as bad credit, and its error rate. Are they the same?"*

In [ ]:
test = Xte.copy(); test['actual'] = yte.values
test['pred'] = clf.predict(Xte)
if 'age' in test.columns:
    bins = pd.cut(test['age'], [0, 25, 54, 200], labels=['under 25', '25–54', '55+'])
    summary = test.groupby(bins).apply(
        lambda t: pd.Series({
            'n': len(t),
            'flagged_bad_rate': round(t['pred'].mean(), 3),
            'error_rate': round((t['pred'] != t['actual']).mean(), 3),
        }))
    print(summary)

## Step 5, The hard question

You've likely found the model is **harder on younger applicants**. Now sit with why:

- Is age *causing* default, or is it standing in for something else (shorter credit history, lower savings) that the data ties to being young?
- The bank's data reflects **past** lending decisions. If those were biased, what has the model learned?
- Is it even *legal* to price a loan on age? (In much of the EU, for many products, no.)

In [ ]:
# Notes: is age a fair basis for this decision? Why / why not?
#

## Step 6, Your verdict (this is the point)

Write the answer you would give the bank's **compliance officer**:

> *"Your model denies credit to a 25-year-old with a small loan and short history, but approves a 55-year-old with identical financials. Why? Is this fair? What do we do about it?"*

A strong answer: states what the model does, explains *why* (the mechanism, not just the number), takes a position on fairness, and proposes a concrete next step, audit, drop the feature, re-collect data, add human review. **"The model said so" is not an answer.**

In [ ]:
# Your answer to the compliance officer:
#

## Before you leave

Jot these down for the debrief and for Session 5:
1. Accuracy vs fairness, how can a model be **good at both** the average and **unfair** to a group?
2. One feature you'd think hard about before letting this model go live, and why.
3. One sentence you'd say to a client who tells you "just use the most accurate model".